<a href="https://colab.research.google.com/github/esprydi/sentimen-analisis-shopee-playstore/blob/master/Scraping_Sentiment_Analisis_shopee_Playstore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install necessary libraries
!pip install google-play-scraper
!pip install sastrawi

# --- Standard Library Imports ---
import re
import string

# --- Third-Party Library Imports ---
import pandas as pd  # Pandas for data manipulation and analysis
pd.options.mode.chained_assignment = None  # Disable chaining warning
import numpy as np  # NumPy for numerical computation
import matplotlib.pyplot as plt  # Matplotlib for data visualization
import seaborn as sns  # Seaborn for statistical data visualization, setting visualization styles
from sklearn.metrics import accuracy_score # For model evaluation
from google_play_scraper import app, reviews, Sort, reviews_all # For scraping Google Play Store data
from wordcloud import WordCloud  # For creating word cloud visualizations

# --- NLTK and Sastrawi Imports for Text Preprocessing ---
import nltk  # Natural Language Toolkit library
from nltk.tokenize import word_tokenize  # For text tokenization
from nltk.corpus import stopwords  # For list of stopwords

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory  # For Indonesian stemming
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory  # For Indonesian stopwords removal

# --- Configuration and Seed for Reproducibility ---
seed = 0
np.random.seed(seed)  # Set seed for reproducibility

# --- NLTK Data Downloads ---
nltk.download('punkt')  # Download data for text tokenization
nltk.download('stopwords')  # Download stopwords list

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 6.3 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

#Scraping Dataset

In [ ]:
#mengimpor pustaka google_play_scrapper

from google_play_scraper import app, reviews_all, Sort

# Mengambil semua ulasan dari aplikasi
scrapreview = reviews(
    'com.shopee.id',
    lang='id',
    country='id',
    sort=Sort.MOST_RELEVANT,
    count=20000
)

# Menyimpan Hasil Scraping

In [ ]:
import pandas as pd

# Membuat DataFrame dari hasil scrapreview
app_reviews_df = pd.DataFrame(scrapreview[0])

# Menyimpan DataFrame ke Google Drive
# Pastikan Google Drive sudah terpasang (mounted) dengan menjalankan kode mount drive di sel terpisah.
output_path = '/content/drive/MyDrive/shopee_app_reviews.csv'
app_reviews_df.to_csv(output_path, index=False, header=True)

print(f"Data ulasan berhasil disimpan ke: {output_path}")

# Menghitung jumlah baris dan kolom dalam DataFrame
jumlah_ulasan, jumlah_kolom = app_reviews_df.shape

# Menampilkan hasil
print(f"Jumlah ulasan: {jumlah_ulasan}")

Jumlah ulasan: 20000


<function print(*args, sep=' ', end='\n', file=None, flush=False)>

In [ ]:
# Menampilkan lima baris pertama
app_reviews_df.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,675d7447-9df2-4eba-85b4-24660d842bb5,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Asli keamanannya bagus, aman juga lancar semua...",5,4,3.68.42,2026-03-03 12:39:10,"Hi kak misterius_, makasih ya buat review bint...",2026-03-03 14:25:07,3.68.42
1,e5fa64c2-fcee-429e-8775-0644f584a59b,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Top,Saya sangat suka belanja disini selain ban...",5,2,3.68.42,2026-03-10 07:31:47,"Hi kak Pipi Papi, maaf ya sudah buat kmu gak n...",2026-03-10 08:21:09,3.68.42
2,faa9dab8-1efb-4209-80b8-2cbdbcd152e9,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Banyak error, ya ampun lemot banget terutama b...",2,1,3.68.42,2026-03-09 11:56:59,"Hai kak Terrysha Cindy Auxilia, makasih ya bua...",2026-03-09 12:48:10,3.68.42
3,9bb6cc1b-6d4c-4c2e-9a9e-97bd14b7125d,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"sistem sering error, saat mesan secara sengaja...",1,0,3.69.34,2026-03-13 08:43:09,"Hai kak Aikawa Miku , terima kasih ya untuk fe...",2026-03-13 09:07:47,3.69.34
4,246efb4d-c63f-4c06-9b30-4fc8dcb8a0e0,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,koneksinya padahal bagus tapi susah banget bua...,5,5,3.68.42,2026-03-06 03:37:59,"Hi Kak Misaki Yuki, maaf ya udah bikin Kakak g...",2026-03-06 06:50:38,3.68.42


In [ ]:
# Menampilkan informasi
app_reviews_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   reviewId              20000 non-null  object        
 1   userName              20000 non-null  object        
 2   userImage             20000 non-null  object        
 3   content               20000 non-null  object        
 4   score                 20000 non-null  int64         
 5   thumbsUpCount         20000 non-null  int64         
 6   reviewCreatedVersion  19854 non-null  object        
 7   at                    20000 non-null  datetime64[ns]
 8   replyContent          18729 non-null  object        
 9   repliedAt             18729 non-null  datetime64[ns]
 10  appVersion            19854 non-null  object        
dtypes: datetime64[ns](2), int64(2), object(7)
memory usage: 1.7+ MB
